# A projected heatwave day in Spain

This example follows one projected day, 24 July 2030. The grid simulation uses
the 13:00 hour. The weather fields are generated from the 2024 historical
heatwave event used in the paper. This notebook displays the resulting weather;
it does not reconstruct the historical-to-future transformation.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "TDOPF_eur.py").is_file())
data = root / "vis/demos/data/spain_2030"
weather = xr.open_dataset(data / "weather_day.nc")
assert weather.sizes["time"] == 24
weather.time.values[13]

## Weather through the day

The curves average over the gridded weather field shown below. They are not
country-wide population-weighted statistics. The vertical line marks the hour
used in the grid example.

In [ ]:
hours = np.arange(24)
fields = [
    ((weather.temperature - 273.15).mean(("x", "y")), "Air temperature (C)", "#c44536"),
    (weather.wnd10m.mean(("x", "y")), "10-m wind speed (m/s)", "#20707c"),
    (weather.influx.mean(("x", "y")), "Solar irradiance (W/m2)", "#c78a28"),
]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for axis, (values, label, colour) in zip(axes, fields):
    axis.plot(hours, values, color=colour, lw=2)
    axis.axvline(13, color="#343a40", ls="--", lw=1)
    axis.set(xlabel="Hour", ylabel=label, xlim=(0, 23))
    axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## The selected hour

These are the weather inputs for the same 13:00 example used in the next two
notebooks. The maps show the model weather grid, not measured conditions in 2030.

In [ ]:
selected = weather.sel(time=np.datetime64("2030-07-24T13:00:00"))
layers = [
    (selected.temperature - 273.15, "Air temperature (C)", "coolwarm"),
    (selected.wnd10m, "10-m wind speed (m/s)", "viridis"),
    (selected.influx, "Solar irradiance (W/m2)", "YlOrRd"),
]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True, sharey=True)
for axis, (values, title, palette) in zip(axes, layers):
    mesh = axis.pcolormesh(selected.lon, selected.lat, values,
                           cmap=palette, shading="auto")
    axis.set(title=title, xlabel="Longitude")
    fig.colorbar(mesh, ax=axis, fraction=0.045, pad=0.02)
axes[0].set_ylabel("Latitude")
fig.tight_layout()
plt.show()

The next notebook shows how these conditions enter demand, generation and
transmission models. It does not treat this illustrative day as a prediction
that a future heatwave will occur on that date.